# Exercise 10 - Dropout and Batch Normalization

The goal of this exercise is to experiment with dropout and batch normalization using a simple CNN and the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset, which is described in detail in the paper [Learning Multiple Layers of Features from Tiny Images](https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf), Alex Krizhevsky, 2009. The [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset consists of 60,000 32x32 pixel colour images in 10 classes, with 6,000 images per class. There are 50,000 training images and 10,000 test images.

- Use **T4 GPU** as hardware accelerator for this exercise

First run the code below to build and train the CNN without dropout or batch normalization to establish a benchmark.



### Step 1:
Download and save CIFAR-10 dataset from Kaggle

In [ ]:
import os
if not os.path.exists('./cifar10-python.zip'):
  !curl -L -o cifar10-python.zip https://www.kaggle.com/api/v1/datasets/download/pankrzysiu/cifar10-python

### Step 2:
Perform the following pre-processing operations for dataset preparation
- Unzip the dataset
- Extract batches of training and test images
- Prepreprocess images to conform to Keras channels-last format (num_images, 32, 32, 3)
- Create x_train, y_train, x_test and y_test arrays
- Print out the shapes of these arrays

In [ ]:
import os
import zipfile
import pickle
import numpy as np
from tensorflow.keras.utils import to_categorical

def extract_cifar10_zip(zip_path, extract_to):
    """Extracts the local zip file into the target directory."""
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction complete.")

    # Locate the folder inside the extracted directory
    # Depending on how it was zipped, files might be in the root or a subfolder
    for root, dirs, files in os.walk(extract_to):
        if "data_batch_1" in files:
            return root
    raise FileNotFoundError("Could not find CIFAR-10 data batches in the extracted files.")

def load_pickle_batch(file_path):
    """Loads a single un-pickled batch file from disk."""
    with open(file_path, 'rb') as f:
        # encoding='latin1' ensures Python 3 compatibility with byte streams
        batch_dict = pickle.load(f, encoding='latin1')
    return batch_dict

def preprocess_images(raw_data):
    """Converts raw flat features into channels-last normalized images."""
    # CIFAR stores channels-first: 3 channels by 1024 pixels (32x32)
    images = raw_data.reshape(-1, 3, 32, 32)
    # Transpose to Keras channels-last layout: (num_images, 32, 32, 3)
    images = images.transpose(0, 2, 3, 1)
    # Cast to float and scale pixel values down to [0.0, 1.0]
    return images.astype('float32') / 255.0

def process_dataset(data_dir):
    """Loops through extracted files to build training and testing sets."""
    x_train_list = []
    y_train_list = []

    # 1. Compile the 5 training data batches
    for i in range(1, 6):
        batch_file = os.path.join(data_dir, f"data_batch_{i}")
        batch = load_pickle_batch(batch_file)
        x_train_list.append(batch['data'])
        y_train_list.extend(batch['labels'])

    x_train_raw = np.concatenate(x_train_list, axis=0)
    y_train = np.array(y_train_list)

    # 2. Compile the single testing batch
    test_file = os.path.join(data_dir, "test_batch")
    test_batch = load_pickle_batch(test_file)
    x_test_raw = test_batch['data']
    y_test = np.array(test_batch['labels'])

    # 3. Shape and scale the image arrays
    x_train = preprocess_images(x_train_raw)
    x_test = preprocess_images(x_test_raw)

    # 4. Transform scalar labels to 10-class one-hot vectors
    y_train = to_categorical(y_train, num_classes=10)
    y_test = to_categorical(y_test, num_classes=10)

    return (x_train, y_train), (x_test, y_test)

if __name__ == "__main__":
    # Ensure this points directly to your local zip archive
    ZIP_FILE_PATH = "./cifar10-python.zip"
    OUTPUT_DIRECTORY = "./extracted_cifar10"

    if not os.path.exists(ZIP_FILE_PATH):
        print(f"Error: Please place '{ZIP_FILE_PATH}' in this directory before running.")
    else:
        # Run conversion pipeline
        data_batch_folder = extract_cifar10_zip(ZIP_FILE_PATH, OUTPUT_DIRECTORY)
        (x_train, y_train), (x_test, y_test) = process_dataset(data_batch_folder)

        # Verify output formats
        print("\n--- Processing Results ---")
        print(f"x_train array: {x_train.shape} | Type: {x_train.dtype}")
        print(f"y_train array: {y_train.shape} | Type: {y_train.dtype}")
        print(f"x_test array:  {x_test.shape} | Type: {x_test.dtype}")
        print(f"y_test array:  {y_test.shape} | Type: {y_test.dtype}")

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import tensorflow
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Activation, MaxPooling2D, Dense, Flatten, Input
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.utils  import to_categorical

# Ensure a clean start
tensorflow.keras.backend.clear_session()

# The data, shuffled and split between train and test sets:
#(x_train, y_train), (x_test, y_test) = cifar10.load_data()


# Plot some images as a sanity check
fig = plt.figure(1, figsize=(15,1))
m = 10
for i in range(m):
    a = fig.add_subplot(1,m,i+1)
    # x_train is already in [0,1] from the previous cell, suitable for imshow.
    plt.imshow(x_train[i])
plt.show()

n_labels = 10

# Normalize the images to have zero mean and values in range [-1,+1]
# x_train and x_test are already float32 and in [0,1] from VeurrvL-HPgR.
# Correct normalization: Scale from [0,1] to [-1,1], then subtract mean.
'''x_train = (x_train * 2.0) - 1.0
x_test  = (x_test * 2.0) - 1.0
mean    = x_train.mean()
x_train = x_train - mean
x_test  = x_test - mean'''

model = Sequential()
model.add(Input(shape=x_train.shape[1:]))
model.add(Conv2D(32, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(Conv2D(32, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dense(n_labels))
model.add(Activation('softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=32, epochs=10,
          validation_data=(x_test, y_test), shuffle=True);

Now add dropout and batch normalization to the model. You should experiment with adding dropout and batch normalization separately so you can see their individual effects as well as trying them both together. You would typically apply dropout at the outputs of layers that have a large number of units and dense connections (which implies a high degree of variance). You can select different degrees of dropout. For example, `Dropout(0.25)` drops a randomly selected 25% of the units in a layer, then multiplies the activations of that layer by 4/3 in the final trained model. You would typically apply batch normalization at the inputs of non-linear activation functions in order to keep the inputs to the non-linearity around the sweet spot.

Which of the two, dropout or batch normalization, has the stronger regularization effect?

In [ ]:
#

In [ ]:
# Here is our answer. Do not run the cell below unless you want to see the answer we provide!
import numpy as np
import matplotlib.pyplot as plt
import tensorflow
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Activation, MaxPooling2D, Dense, Flatten
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.utils  import to_categorical
from tensorflow.keras.backend  import clear_session

#This function uses x_train, y_train, x_test and y_test from the cell above

def build_and_run_model(x_train_input, y_train_input, x_test_input, y_test_input, dropout = False, bn = False):

    # Ensure a clean start
    clear_session()

    n_labels = 10

    x_train_final = x_train_input
    x_test_final = x_test_input

    # Use the preprocessed y data directly
    y_train_final = y_train_input
    y_test_final = y_test_input

    model = Sequential()
    model.add(Input(shape=x_train_final.shape[1:]))
    model.add(Conv2D(32, (3, 3), padding='same',  use_bias=(not bn)))
    if bn: model.add(BatchNormalization(scale=False))
    model.add(Activation('relu'))
    model.add(Conv2D(32, (3, 3), padding='same', use_bias=(not bn)))
    if bn: model.add(BatchNormalization(scale=False))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(64, (3, 3), padding='same', use_bias=(not bn)))
    if bn: model.add(BatchNormalization(scale=False))
    model.add(Activation('relu'))
    model.add(Conv2D(64, (3, 3), use_bias=(not bn)))
    if bn: model.add(BatchNormalization(scale=False))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    if dropout: model.add(Dropout(0.5))

    model.add(Flatten())
    model.add(Dense(512, use_bias=(not bn)))
    if bn: model.add(BatchNormalization(scale=False))
    model.add(Activation('relu'))
    if dropout: model.add(Dropout(0.5))

    model.add(Dense(n_labels))
    model.add(Activation('softmax'))
    model.summary()

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(x_train_final, y_train_final,
              batch_size=32, epochs=10,
              validation_data=(x_test_final, y_test_final), shuffle=True);

print('\nDropout only')
build_and_run_model(x_train, y_train, x_test, y_test, dropout = True,  bn = False)

print('\nBatch normalization only')
build_and_run_model(x_train, y_train, x_test, y_test, dropout = False, bn = True)

print('\nBoth dropout and batch normalization')
build_and_run_model(x_train, y_train, x_test, y_test, dropout = True,  bn = True)
